# Сбор обучающего датасета для прогноза количества операций

Цель: подготовить датасеты для прогноза **количества операций на следующий месяц**.

Шаги:
1) Собрать базовый датасет только из истории операций (`operations_from_history.csv`).
2) Добавить внешний регрессор `workDayMonth_2` (кол-во рабочих дней) из `calculate_parameter`.

Результаты:
- `data/processed/operations_training_dataset_base.csv`
- `data/processed/operations_training_dataset_wdm2.csv`


In [ ]:
import json
from pathlib import Path

import pandas as pd

PROCESSED_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")

OPS_PATH = PROCESSED_DIR / "operations_from_history.csv"
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

OPS_PATH, RAW_FILES

In [ ]:
# 1) Базовый датасет: только period + operation_fact_value

df_ops = pd.read_csv(OPS_PATH, parse_dates=["period"])

# нормализуем период до месяца

df_ops["period"] = df_ops["period"].dt.to_period("M").dt.to_timestamp()

# агрегируем по месяцам (на случай нескольких записей в месяц)

df_ops_month = (
    df_ops.groupby(["resist_id", "operation_id", "period"], as_index=False)
    .agg({"operation_fact_value": "sum"})
)

# цель: предсказать следующий месяц

df_ops_month = df_ops_month.sort_values(["resist_id", "operation_id", "period"])

df_ops_month["target_next_month"] = (
    df_ops_month.groupby(["resist_id", "operation_id"])["operation_fact_value"]
    .shift(-1)
)

# признаки минимальные: текущий факт + период

df_ops_month["month_num"] = df_ops_month["period"].dt.month

df_ops_month["year_num"] = df_ops_month["period"].dt.year

# удаляем строки без следующего месяца

df_base = df_ops_month.dropna(subset=["target_next_month"]).copy()

out_base = PROCESSED_DIR / "operations_training_dataset_base.csv"

# сохраняем

df_base.to_csv(out_base, index=False)

print("Base dataset saved:", out_base)
print("Rows:", len(df_base))

df_base.head()

In [ ]:
# 2) Добавляем внешний регрессор workDayMonth_2 из calculate_parameter

def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def extract_workdaymonth(calc_json: str):
    if pd.isna(calc_json):
        return None
    try:
        obj = json.loads(calc_json)
    except Exception:
        return None
    v = obj.get("workDayMonth_2")
    if isinstance(v, dict):
        return v.get("value")
    return None


rows = []
for p in RAW_FILES:
    part = pd.read_csv(p, dtype=str)
    part["period"] = pd.to_datetime(part["period"])

    # фильтр согласования (approved)
    part["send"] = to_bool(part["send"])
    part["cancel_approve"] = to_bool(part["cancel_approve"])
    part["validate_one"] = to_bool(part["validate_one"])
    part["validate_two"] = to_bool(part["validate_two"])
    part["validate_count"] = pd.to_numeric(part["validate_count"], errors="coerce").fillna(0).astype(int)

    mask_sent = part["send"]
    mask_not_cancelled = ~part["cancel_approve"]
    mask_approved = (
        (part["validate_count"] == 0)
        | ((part["validate_count"] == 1) & part["validate_one"])
        | ((part["validate_count"] >= 2) & part["validate_one"] & part["validate_two"])
    )

    part_ok = part[mask_sent & mask_not_cancelled & mask_approved].copy()

    part_ok["workDayMonth_2"] = part_ok["calculate_parameter"].apply(extract_workdaymonth)
    part_ok["workDayMonth_2"] = pd.to_numeric(part_ok["workDayMonth_2"], errors="coerce")

    part_ok["period"] = part_ok["period"].dt.to_period("M").dt.to_timestamp()

    rows.append(
        part_ok[["resist_id", "period", "workDayMonth_2"]]
        .dropna(subset=["resist_id", "period"])
    )


df_wdm = pd.concat(rows, ignore_index=True)

# агрегируем на уровень месяца

df_wdm = (
    df_wdm.groupby(["resist_id", "period"], as_index=False)
    .agg({"workDayMonth_2": "mean"})
)

# объединяем с базовым датасетом

df_wdm2 = df_base.merge(df_wdm, on=["resist_id", "period"], how="left")

out_wdm2 = PROCESSED_DIR / "operations_training_dataset_wdm2.csv"

df_wdm2.to_csv(out_wdm2, index=False)

print("WDM2 dataset saved:", out_wdm2)
print("Rows:", len(df_wdm2))

df_wdm2.head()

# Сбор обучающего датасета по операциям (operations)

Этот ноутбук готовит датасет для прогноза **количества операций на следующий месяц**.

Вход: `data/processed/operations_from_history.csv`
Выход: `data/processed/operations_training_dataset.csv`


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
OPS_PATH = PROCESSED_DIR / "operations_from_history.csv"
OUT_PATH = PROCESSED_DIR / "operations_training_dataset.csv"

OPS_PATH

In [ ]:
# Загрузка и нормализация (load & normalize)

df = pd.read_csv(OPS_PATH, parse_dates=["period"])

# Нормализуем период до месяца

df["period"] = df["period"].dt.to_period("M").dt.to_timestamp()

# Чистим типы

df["resist_id"] = pd.to_numeric(df["resist_id"], errors="coerce").astype("Int64")
df["operation_id"] = pd.to_numeric(df["operation_id"], errors="coerce").astype("Int64")
df["operation_fact_value"] = pd.to_numeric(df["operation_fact_value"], errors="coerce").fillna(0)

# Убираем пустые ключи

df = df.dropna(subset=["resist_id", "operation_id", "period"])

# Аггрегируем на уровень: операция × подразделение × месяц

df_m = (
    df.groupby(["resist_id", "operation_id", "period"], as_index=False)
    .agg({
        "operation_fact_value": "sum",
        "division_title": "first",
    })
)

print("Строк в monthly датасете:", len(df_m))
df_m.head()

In [ ]:
# Признаки (features) и таргет (target = следующий месяц)

df_m = df_m.sort_values(["resist_id", "operation_id", "period"]).copy()

g = df_m.groupby(["resist_id", "operation_id"], group_keys=False)

# Лаги (lags)
for lag in [1, 2, 3, 12]:
    df_m[f"lag_{lag}"] = g["operation_fact_value"].shift(lag)

# Rolling stats (rolling stats) только по прошлому

df_m["roll_mean_3"] = g["operation_fact_value"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).mean()
)

df_m["roll_med_3"] = g["operation_fact_value"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).median()
)

# Сезонность (seasonality)

df_m["month_num"] = df_m["period"].dt.month
df_m["year_num"] = df_m["period"].dt.year

# Таргет: значение следующего месяца

df_m["target_next_month"] = g["operation_fact_value"].shift(-1)

# Убираем строки без таргета

df_train = df_m.dropna(subset=["target_next_month"]).copy()

print("Строк в train датасете:", len(df_train))
df_train.head()

In [ ]:
# Сохранение (export)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(OUT_PATH, index=False)

print("Файл сохранен:", OUT_PATH)
print("Колонки:", list(df_train.columns))